In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.customer_sow_comparison AS
SELECT
    Customer_ID,

    MAX(CASE WHEN Fiscal_Year = 2025 THEN Total_Spend END)
        AS FY2025_Total_Spend,

    MAX(CASE WHEN Fiscal_Year = 2025 THEN HSIC_Spend END)
        AS FY2025_HSIC_Spend,

    MAX(CASE WHEN Fiscal_Year = 2025 THEN HSIC_SoW END)
        AS FY2025_SoW,

    MAX(CASE WHEN Fiscal_Year = 2026 THEN Total_Spend END)
        AS FY2026_Total_Spend,

    MAX(CASE WHEN Fiscal_Year = 2026 THEN HSIC_Spend END)
        AS FY2026_HSIC_Spend,

    MAX(CASE WHEN Fiscal_Year = 2026 THEN HSIC_SoW END)
        AS FY2026_SoW

FROM synchrony.analytics.customer_fy_sow
GROUP BY Customer_ID;

In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.customer_sow_comparison_final AS
SELECT
    *,
    
    FY2026_SoW - FY2025_SoW AS SoW_Change,
    
    FY2026_HSIC_Spend - FY2025_HSIC_Spend AS HSIC_Spend_Change,
    
    FY2026_Total_Spend - FY2025_Total_Spend AS Total_Spend_Change

FROM synchrony.analytics.customer_sow_comparison;

In [0]:
%sql
SELECT
    COUNT(*) AS Total_Customers,

    SUM(
        CASE
            WHEN FY2025_SoW IS NOT NULL
             AND FY2026_SoW IS NOT NULL
            THEN 1 ELSE 0
        END
    ) AS Comparable_Customers,

    SUM(
        CASE
            WHEN SoW_Change < 0
            THEN 1 ELSE 0
        END
    ) AS SoW_Decliners,

    SUM(
        CASE
            WHEN SoW_Change > 0
            THEN 1 ELSE 0
        END
    ) AS SoW_Increasers,

    SUM(
        CASE
            WHEN SoW_Change = 0
            THEN 1 ELSE 0
        END
    ) AS No_SoW_Change

FROM synchrony.analytics.customer_sow_comparison_final;

In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.customer_movement_analysis AS
SELECT
    *,

    CASE
        WHEN SoW_Change < 0 THEN 'SoW_DECLINED'
        WHEN SoW_Change > 0 THEN 'SoW_INCREASED'
        WHEN SoW_Change = 0 THEN 'NO_CHANGE'
        ELSE 'NOT_COMPARABLE'
    END AS SoW_Movement,

    CASE
        WHEN HSIC_Spend_Change < 0 THEN 'HSIC_SPEND_DOWN'
        WHEN HSIC_Spend_Change > 0 THEN 'HSIC_SPEND_UP'
        WHEN HSIC_Spend_Change = 0 THEN 'HSIC_NO_CHANGE'
        ELSE 'NO_DATA'
    END AS HSIC_Movement,

    CASE
        WHEN Total_Spend_Change < 0 THEN 'TOTAL_SPEND_DOWN'
        WHEN Total_Spend_Change > 0 THEN 'TOTAL_SPEND_UP'
        WHEN Total_Spend_Change = 0 THEN 'TOTAL_NO_CHANGE'
        ELSE 'NO_DATA'
    END AS Total_Spend_Movement

FROM synchrony.analytics.customer_sow_comparison_final;

In [0]:
%sql
SELECT
    SoW_Movement,
    HSIC_Movement,
    Total_Spend_Movement,
    COUNT(*) AS Customers
FROM synchrony.analytics.customer_movement_analysis
WHERE SoW_Movement <> 'NOT_COMPARABLE'
GROUP BY
    SoW_Movement,
    HSIC_Movement,
    Total_Spend_Movement
ORDER BY
    SoW_Movement,
    Customers DESC;

In [0]:
%sql
SELECT
    c.Customer_ID,

    SUM(CASE
        WHEN m.Fiscal_Year = 2025 THEN m.HSIC_Spend ELSE 0
    END) AS FY2025_HSIC,

    SUM(CASE
        WHEN m.Fiscal_Year = 2026 THEN m.HSIC_Spend ELSE 0
    END) AS FY2026_HSIC,

    SUM(CASE
        WHEN m.Fiscal_Year = 2025 THEN m.Competitor_Card_Spend ELSE 0
    END) AS FY2025_Competitor_Card,

    SUM(CASE
        WHEN m.Fiscal_Year = 2026 THEN m.Competitor_Card_Spend ELSE 0
    END) AS FY2026_Competitor_Card,

    SUM(CASE
        WHEN m.Fiscal_Year = 2025 THEN m.Debit_Card_Spend ELSE 0
    END) AS FY2025_Debit,

    SUM(CASE
        WHEN m.Fiscal_Year = 2026 THEN m.Debit_Card_Spend ELSE 0
    END) AS FY2026_Debit,

    SUM(CASE
        WHEN m.Fiscal_Year = 2025 THEN m.Cash_UPI_Spend ELSE 0
    END) AS FY2025_Cash_UPI,

    SUM(CASE
        WHEN m.Fiscal_Year = 2026 THEN m.Cash_UPI_Spend ELSE 0
    END) AS FY2026_Cash_UPI,

    SUM(CASE
        WHEN m.Fiscal_Year = 2025 THEN m.Wallet_Spend ELSE 0
    END) AS FY2025_Wallet,

    SUM(CASE
        WHEN m.Fiscal_Year = 2026 THEN m.Wallet_Spend ELSE 0
    END) AS FY2026_Wallet

FROM synchrony.analytics.customer_movement_analysis c
JOIN synchrony.analytics.customer_monthly_sow m
    ON c.Customer_ID = m.Customer_ID

WHERE c.SoW_Movement = 'SoW_DECLINED'
  AND c.HSIC_Movement = 'HSIC_SPEND_DOWN'
  AND c.Total_Spend_Movement = 'TOTAL_SPEND_UP'

GROUP BY c.Customer_ID;

In [0]:
%sql
SELECT
    SUM(FY2025_HSIC) AS FY2025_HSIC,
    SUM(FY2026_HSIC) AS FY2026_HSIC,

    SUM(FY2025_Competitor_Card) AS FY2025_Competitor_Card,
    SUM(FY2026_Competitor_Card) AS FY2026_Competitor_Card,

    SUM(FY2025_Debit) AS FY2025_Debit,
    SUM(FY2026_Debit) AS FY2026_Debit,

    SUM(FY2025_Cash_UPI) AS FY2025_Cash_UPI,
    SUM(FY2026_Cash_UPI) AS FY2026_Cash_UPI,

    SUM(FY2025_Wallet) AS FY2025_Wallet,
    SUM(FY2026_Wallet) AS FY2026_Wallet

FROM (
    SELECT
        c.Customer_ID,

        SUM(CASE WHEN m.Fiscal_Year = 2025
                 THEN m.HSIC_Spend ELSE 0 END) AS FY2025_HSIC,
        SUM(CASE WHEN m.Fiscal_Year = 2026
                 THEN m.HSIC_Spend ELSE 0 END) AS FY2026_HSIC,

        SUM(CASE WHEN m.Fiscal_Year = 2025
                 THEN m.Competitor_Card_Spend ELSE 0 END) AS FY2025_Competitor_Card,
        SUM(CASE WHEN m.Fiscal_Year = 2026
                 THEN m.Competitor_Card_Spend ELSE 0 END) AS FY2026_Competitor_Card,

        SUM(CASE WHEN m.Fiscal_Year = 2025
                 THEN m.Debit_Card_Spend ELSE 0 END) AS FY2025_Debit,
        SUM(CASE WHEN m.Fiscal_Year = 2026
                 THEN m.Debit_Card_Spend ELSE 0 END) AS FY2026_Debit,

        SUM(CASE WHEN m.Fiscal_Year = 2025
                 THEN m.Cash_UPI_Spend ELSE 0 END) AS FY2025_Cash_UPI,
        SUM(CASE WHEN m.Fiscal_Year = 2026
                 THEN m.Cash_UPI_Spend ELSE 0 END) AS FY2026_Cash_UPI,

        SUM(CASE WHEN m.Fiscal_Year = 2025
                 THEN m.Wallet_Spend ELSE 0 END) AS FY2025_Wallet,
        SUM(CASE WHEN m.Fiscal_Year = 2026
                 THEN m.Wallet_Spend ELSE 0 END) AS FY2026_Wallet

    FROM synchrony.analytics.customer_movement_analysis c
    JOIN synchrony.analytics.customer_monthly_sow m
        ON c.Customer_ID = m.Customer_ID

    WHERE c.SoW_Movement = 'SoW_DECLINED'
      AND c.HSIC_Movement = 'HSIC_SPEND_DOWN'
      AND c.Total_Spend_Movement = 'TOTAL_SPEND_UP'

    GROUP BY c.Customer_ID
) x;